In [ ]:
import os
import time
import psycopg2
from multiprocessing import Pool, cpu_count
from utils import DB_PARAMS
from sqlalchemy import create_engine
from sqlalchemy.types import UserDefinedType
from sqlalchemy.orm import declarative_base
from sqlalchemy import Column, Computed, Integer, String, Index
from geoalchemy2 import Geometry
import psycopg2

In [ ]:
# Replace with your PostgreSQL credentials
username = DB_PARAMS['user']
password = DB_PARAMS['password']
host = DB_PARAMS['host']  # or your database host
port = DB_PARAMS['port']       # default PostgreSQL port
database = DB_PARAMS['database']
# Create the connection string
connection_string = f'postgresql://{username}:{password}@{host}:{port}/{database}'

# Create the SQLAlchemy engine
engine = create_engine(connection_string)

# Test the connection
try:
    with engine.connect() as connection:
        print("Connection to PostgreSQL database successful!")
except Exception as e:
    print(f"An error occurred: {e}")

Connection to PostgreSQL database successful!


In [4]:
# Custom H3Index type
class H3Index(UserDefinedType):
    cache_ok = True
    
    def get_col_spec(self):
        return "H3INDEX"
    
    def bind_processor(self, dialect):
        def process(value):
            return value
        return process
    
    def result_processor(self, dialect, coltype):
        def process(value):
            return value
        return process

In [ ]:
Base = declarative_base()
class Patch(Base):
    __tablename__ = 'patches'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    slide_name = Column(String(255))
    geom = Column(Geometry('POINT', srid=4326), nullable=False)
    h3_index = Column(
        H3Index,
        Computed('h3_lat_lng_to_cell(geom, 9)', persisted=True)
    )
    
    # Define indexes
    __table_args__ = (
        Index('idx_patches_h3', 'h3_index'),
    )
    
    def __repr__(self):
        return f"<Patch(id={self.id}, h3={self.h3_index})>"
    

# Create the table in the database
Base.metadata.create_all(engine)
print("Table 'patches' created successfully.")


Table 'patches' created successfully.


In [ ]:
# Configuration
NUM_WORKERS = cpu_count()  # or set manually, e.g., 8
CHUNK_SIZE = 10_000_000  # 10M rows per file
TOTAL_ROWS = 1_000_000_000  # 1 billion rows

OUTPUT_DIR = './bulk_load_chunks'

# Step 1: Generate CSV files in parallel
def generate_chunk_file(args):
    """Generate a single CSV file for a chunk of data"""
    chunk_id, start_idx, end_idx = args
    
    filename = f"{OUTPUT_DIR}/chunk_{chunk_id:04d}.csv"
    
    with open(filename, 'w') as f:
        for i in range(start_idx, end_idx):
            # Replace this with your actual data generation logic
            slide_name = f"slide_{i}"
            lon = -180 + (i % 360)
            lat = -90 + (i % 180)
            
            f.write(f"{slide_name}\tSRID=4326;POINT({lon} {lat})\n")
    
    print(f"Generated chunk {chunk_id}: {end_idx - start_idx:,} rows")
    return filename

def generate_files_parallel(total_rows, num_workers=NUM_WORKERS, chunk_size=CHUNK_SIZE):
    """Generate all CSV files in parallel"""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Calculate chunks
    chunks = []
    for i in range(0, total_rows, chunk_size):
        chunk_id = len(chunks)
        start_idx = i
        end_idx = min(i + chunk_size, total_rows)
        chunks.append((chunk_id, start_idx, end_idx))
    
    print(f"Generating {len(chunks)} files using {num_workers} workers...")
    start_time = time.time()
    
    with Pool(num_workers) as pool:
        filenames = pool.map(generate_chunk_file, chunks)
    
    elapsed = time.time() - start_time
    print(f"File generation complete: {elapsed:.2f}s ({total_rows/elapsed:,.0f} rows/sec)")
    
    return filenames

# Step 2: Load files into database in parallel
def load_chunk_file(args):
    """Load a single CSV file into the database"""
    filename, worker_id, db_config = args
    
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    
    start_time = time.time()
    
    with open(filename, 'r') as f:
        cur.copy_expert(
            """
            COPY patches (slide_name, geom) 
            FROM STDIN 
            WITH (FORMAT CSV, DELIMITER E'\t')
            """,
            f
        )
    
    conn.commit()
    elapsed = time.time() - start_time
    
    # Count rows for reporting
    with open(filename, 'r') as f:
        row_count = sum(1 for _ in f)
    
    print(f"Worker {worker_id} loaded {filename}: {row_count:,} rows in {elapsed:.2f}s ({row_count/elapsed:,.0f} rows/sec)")
    
    cur.close()
    conn.close()
    
    return row_count

def load_files_parallel(filenames, db_config, num_workers=NUM_WORKERS):
    """Load all CSV files into database in parallel"""
    print(f"\nLoading {len(filenames)} files using {num_workers} workers...")
    start_time = time.time()
    
    # Prepare arguments for each worker
    args = [(filename, i, db_config) for i, filename in enumerate(filenames)]
    
    with Pool(num_workers) as pool:
        row_counts = pool.map(load_chunk_file, args)
    
    total_rows = sum(row_counts)
    elapsed = time.time() - start_time
    
    print(f"\nLoad complete: {total_rows:,} rows in {elapsed:.2f}s ({total_rows/elapsed:,.0f} rows/sec)")
    
    return total_rows

# Step 3: Main execution
def bulk_load_billion_rows():
    """Complete pipeline for loading 1 billion rows"""
    
    total_rows = TOTAL_ROWS
    
    print("=" * 80)
    print("STEP 0: Preparing database")
    print("=" * 80)
    
    step_start_time = time.time()
    
    # Drop indexes and optimize for bulk load
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor()
    
    cur.execute("DROP INDEX IF EXISTS idx_patches_h3;")
    cur.execute("ALTER TABLE patches SET (autovacuum_enabled = false);")
    cur.execute("ALTER TABLE patches SET UNLOGGED;")  # Optional: faster but not crash-safe
    cur.execute("SET maintenance_work_mem = '2GB';")
    
    conn.commit()
    cur.close()
    conn.close()
    
    step_elapsed = time.time() - step_start_time
    print(f"Database prepared for bulk load in {step_elapsed:.2f}s\n")
    
    print("=" * 80)
    print("STEP 1: Generating CSV files")
    print("=" * 80)
    
    step_start_time = time.time()
    filenames = generate_files_parallel(total_rows)
    step_elapsed = time.time() - step_start_time
    print(f"CSV file generation completed in {step_elapsed:.2f}s\n")
    
    print("\n" + "=" * 80)
    print("STEP 2: Loading files into database")
    print("=" * 80)
    
    step_start_time = time.time()
    load_files_parallel(filenames, DB_PARAMS)
    step_elapsed = time.time() - step_start_time
    print(f"File loading completed in {step_elapsed:.2f}s\n")
    
    print("\n" + "=" * 80)
    print("STEP 3: Post-load optimization")
    print("=" * 80)
    
    step_start_time = time.time()
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor()
    
    print("Making table logged...")
    cur.execute("ALTER TABLE patches SET LOGGED;")
    conn.commit()
    
    print("Recreating indexes...")
    cur.execute("CREATE INDEX idx_patches_h3 ON patches (h3_index);")
    conn.commit()
    
    print("Re-enabling autovacuum...")
    cur.execute("ALTER TABLE patches SET (autovacuum_enabled = true);")
    conn.commit()
    
    print("Running ANALYZE...")
    cur.execute("ANALYZE patches;")
    conn.commit()
    
    cur.close()
    conn.close()
    
    step_elapsed = time.time() - step_start_time
    print(f"Post-load optimization completed in {step_elapsed:.2f}s\n")
    
    print("\n" + "=" * 80)
    print("COMPLETE!")
    print("=" * 80)
    
    # Cleanup files (optional)
    # import shutil
    # shutil.rmtree(OUTPUT_DIR)

In [ ]:
bulk_load_billion_rows()

# Benchmark the time taken to get a single random item from the table


In [ ]:
import time
from sqlalchemy.orm import sessionmaker
import random


Session = sessionmaker(bind=engine)
session = Session()

# Test with a random ID (adjust based on your data)
test_id = random.randint(1, 1000000)  # Assuming some data exists

print(f"Querying object with ID: {test_id}")

start_time = time.time()
result = session.query(Patch).filter(Patch.id == test_id).first()
end_time = time.time()

query_time = (end_time - start_time) * 1000  # Convert to milliseconds

if result:
    print(f"✓ Found object: {result}")
    print(f"Query time: {query_time:.4f} ms")
else:
    print(f"No object found with ID {test_id}")
    print(f"Query time: {query_time:.4f} ms")

session.close()

In [ ]:
# Test 3: Query First Object within a Specific H3 Hexagon
import time
from sqlalchemy.orm import sessionmaker
from sqlalchemy import text

Session = sessionmaker(bind=engine)
session = Session()

# # Generate a test coordinate and get its H3 index
# test_lat = random.uniform(-90, 90)
# test_lon = random.uniform(-180, 180)

# # Get H3 index for this coordinate at resolution 9
# result = session.execute(
#     text("SELECT h3_lat_lng_to_cell(ST_SetSRID(ST_MakePoint(:lon, :lat), 4326), 9) as h3_index"),
#     {"lat": test_lat, "lon": test_lon}
# )
# test_h3_index = result.fetchone()[0]

# print(f"Testing query for H3 hexagon: {test_h3_index}")
# print(f"(Generated from coordinates: lat={test_lat:.6f}, lon={test_lon:.6f})")

# Query for the first object in this H3 hexagon
start_time = time.time()
result = session.query(Patch).filter(Patch.h3_index == "89071318b83ffff").all()
end_time = time.time()

query_time = (end_time - start_time) * 1000  # Convert to milliseconds

if result:
    print(f"✓ Found object in hexagon: {result}")
    print(f"Query time: {query_time:.4f} ms")
else:
    print(f"No objects found in H3 hexagon 89071318b83ffff")
    print(f"Query time: {query_time:.4f} ms")
    print("Note: This hexagon might be empty. Try running the test again or with more data.")

session.close()

In [ ]:
import time
from sqlalchemy.orm import sessionmaker
from sqlalchemy import text
import random

# Test: Modify a single record (change the geom)

Session = sessionmaker(bind=engine)
session = Session()

# Test with a random ID (adjust based on your data)
test_id = random.randint(1, 1000000)  # Assuming some data exists
new_geom = "SRID=4326;POINT(-122.4194 37.7749)"  # Example: San Francisco coordinates

print(f"Updating object with ID: {test_id}")

start_time = time.time()
result = session.query(Patch).filter(Patch.id == test_id).first()

if result:
    result.geom = new_geom
    session.commit()
    end_time = time.time()
    update_time = (end_time - start_time) * 1000  # Convert to milliseconds
    print(f"✓ Updated object: {result}")
    print(f"Update time: {update_time:.4f} ms")
else:
    end_time = time.time()
    update_time = (end_time - start_time) * 1000  # Convert to milliseconds
    print(f"No object found with ID {test_id}")
    print(f"Update time: {update_time:.4f} ms")

session.close()

Updating object with ID: 109444
✓ Updated object: <Patch(id=109444, h3=89283082803ffff)>
Update time: 9.9399 ms


In [ ]:

# Check if the index exists and is valid
conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

# 1. Check if index exists
print("=== Index Status ===")
cur.execute("""
    SELECT 
        schemaname,
        tablename,
        indexname,
        indexdef
    FROM pg_indexes 
    WHERE tablename = 'patches' 
    AND indexname = 'idx_patches_h3';
""")

result = cur.fetchone()
if result:
    print(f"✓ Index exists: {result[2]}")
    print(f"  Definition: {result[3]}")
else:
    print("✗ Index does not exist!")



cur.close()
conn.close()

=== Index Status ===
✓ Index exists: idx_patches_h3
  Definition: CREATE INDEX idx_patches_h3 ON public.patches USING btree (h3_index)
